<a href="https://colab.research.google.com/github/rfaoktvian/Dicoding_FinalProject/blob/main/Nazly_Rafa_Oktafian_Nuzqu_Proyek_Klasifikasi_Gambar_Dicoding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Latar Belakang

Proyek ini bertujuan membangun model *Convolutional Neural Network* (CNN) untuk mengklasifikasikan gambar pemandangan alam menggunakan **Intel Image Classification Dataset**. Dataset ini terdiri dari sekitar 25.000 gambar berukuran 150×150 piksel yang terbagi ke dalam 6 kategori: *buildings*, *forest*, *glacier*, *mountain*, *sea*, dan *street*.

Model dibangun menggunakan arsitektur Sequential dengan lapisan Conv2D dan Pooling, kemudian dievaluasi pada data train dan test dengan target akurasi minimal 85%. Model akhir akan disimpan dalam format SavedModel, TF-Lite, dan TensorFlow.js untuk keperluan deployment di berbagai platform.

## IMPORT LIBRARY

In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Flatten,
                                     Dense, Dropout, BatchNormalization)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from google.colab import drive

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.19.0


## LOAD DATASET

In [3]:
drive.mount('/content/drive')
base_dir   = '/content/drive/MyDrive/Dicoding/FinalProject/ProyeksiGambar/Intel_Classification/'

def detect_path(base, folder):
    nested = os.path.join(base, folder, folder)
    if os.path.isdir(nested):
        return nested
    direct = os.path.join(base, folder)
    if os.path.isdir(direct):
        return direct
    raise FileNotFoundError(f"Folder '{folder}' tidak ditemukan di {base}")

TRAIN_PATH = detect_path(base_dir, 'seg_train')
TEST_PATH  = detect_path(base_dir, 'seg_test')

print("TRAIN_PATH:", TRAIN_PATH)
print("TEST_PATH :", TEST_PATH)

Mounted at /content/drive
TRAIN_PATH: /content/drive/MyDrive/Dicoding/FinalProject/ProyeksiGambar/Intel_Classification/seg_train/seg_train
TEST_PATH : /content/drive/MyDrive/Dicoding/FinalProject/ProyeksiGambar/Intel_Classification/seg_test/seg_test


In [4]:
print("\nStruktur folder:")
for item in sorted(os.listdir(TRAIN_PATH)):
    count = len(os.listdir(os.path.join(TRAIN_PATH, item)))
    print(f"  {item}/  → {count} gambar")


Struktur folder:
  buildings/  → 2191 gambar
  forest/  → 2271 gambar
  glacier/  → 2404 gambar
  mountain/  → 2512 gambar
  sea/  → 2274 gambar
  street/  → 2382 gambar


In [5]:
IMG_SIZE   = (150, 150)
BATCH_SIZE = 32
SEED       = 42
EPOCHS     = 30

CLASS_NAMES = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
NUM_CLASSES = len(CLASS_NAMES)

print(f"Jumlah kelas  : {NUM_CLASSES}")
print(f"Ukuran gambar : {IMG_SIZE}")
print(f"Batch size    : {BATCH_SIZE}")
print(f"Max epochs    : {EPOCHS}")

Jumlah kelas  : 6
Ukuran gambar : (150, 150)
Batch size    : 32
Max epochs    : 30


In [6]:
train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_PATH,
    validation_split=0.2,
    subset='training',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)
val_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_PATH,
    validation_split=0.2,
    subset='validation',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    TEST_PATH,
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False
)

print(f"\nKelas terdeteksi : {train_dataset.class_names}")
print(f"Batch train      : {len(train_dataset)}")
print(f"Batch val        : {len(val_dataset)}")
print(f"Batch test       : {len(test_dataset)}")

Found 14034 files belonging to 6 classes.
Using 11228 files for training.
Found 14034 files belonging to 6 classes.
Using 2806 files for validation.
Found 3000 files belonging to 6 classes.

Kelas terdeteksi : ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
Batch train      : 351
Batch val        : 88
Batch test       : 94
